In [3]:
# Task 27: Create a SparkSession to start working with distributed data using PySpark.

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NOPIS_SP1_Ingestion")
    .master("local[*]")
    .getOrCreate()
)

print("SparkSession created successfully")

c:\Users\nalin.karthik\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


SparkSession created successfully


In [4]:
print(spark)
print("Spark version:", spark.version)

Spark version: 4.2.0


In [5]:
# Task 28: Import the Spark data types needed to define the CSV schema.

from pyspark.sql.types import (
    StructType,
    StructField,
    TimestampType,
    IntegerType,
    DoubleType
)

In [6]:
# Task 28: Define the expected schema for the raw telecom CSV files.

raw_schema = StructType([
    StructField("datetime", TimestampType(), True),
    StructField("CellID", IntegerType(), True),
    StructField("countrycode", IntegerType(), True),
    StructField("smsin", DoubleType(), True),
    StructField("smsout", DoubleType(), True),
    StructField("callin", DoubleType(), True),
    StructField("callout", DoubleType(), True),
    StructField("internet", DoubleType(), True)
])

In [7]:
# Task 28: Find all Milan daily files using the exact sms-call-internet-mi-*.csv pattern.

from pathlib import Path

data_folder = Path(r"D:\NOPIS\data")

files = [
    str(file)
    for file in data_folder.glob("sms-call-internet-mi-*.csv")
]

print("Files found:", len(files))

for file in files:
    print(file)

Files found: 7
D:\NOPIS\data\sms-call-internet-mi-2013-11-01.csv
D:\NOPIS\data\sms-call-internet-mi-2013-11-02.csv
D:\NOPIS\data\sms-call-internet-mi-2013-11-03.csv
D:\NOPIS\data\sms-call-internet-mi-2013-11-04.csv
D:\NOPIS\data\sms-call-internet-mi-2013-11-05.csv
D:\NOPIS\data\sms-call-internet-mi-2013-11-06.csv
D:\NOPIS\data\sms-call-internet-mi-2013-11-07.csv


In [8]:
# Task 28: Read all Milan daily CSV files together into one distributed Spark DataFrame.

raw_network_df = (
    spark.read
    .option("header", True)
    .schema(raw_schema)
    .csv(files)
)

In [9]:
# Task 28: Check the schema and preview the data loaded from all daily files.

raw_network_df.printSchema()

raw_network_df.show(5, truncate=False)

root
 |-- datetime: timestamp (nullable = true)
 |-- CellID: integer (nullable = true)
 |-- countrycode: integer (nullable = true)
 |-- smsin: double (nullable = true)
 |-- smsout: double (nullable = true)
 |-- callin: double (nullable = true)
 |-- callout: double (nullable = true)
 |-- internet: double (nullable = true)

+-------------------+------+-----------+------+------+------+-------+--------+
|datetime           |CellID|countrycode|smsin |smsout|callin|callout|internet|
+-------------------+------+-----------+------+------+------+-------+--------+
|2013-11-01 00:00:00|1     |0          |0.3521|NULL  |NULL  |0.0273 |NULL    |
|2013-11-01 00:00:00|1     |33         |NULL  |NULL  |NULL  |NULL   |0.0261  |
|2013-11-01 00:00:00|1     |39         |1.7322|1.1047|0.5919|0.402  |57.7729 |
|2013-11-01 00:00:00|2     |0          |0.3581|NULL  |NULL  |0.0273 |NULL    |
|2013-11-01 00:00:00|2     |33         |NULL  |NULL  |NULL  |NULL   |0.0274  |
+-------------------+------+-----------+----

In [10]:
# Task 29: Read the same CSV files using Spark's inferSchema option to automatically detect data types.

inferred_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(files)
)

In [11]:
# Task 29: Compare the manually defined schema with the schema detected automatically by Spark.

print("--- Manual Schema ---")
raw_network_df.printSchema()

print("--- Inferred Schema ---")
inferred_df.printSchema()

--- Manual Schema ---
root
 |-- datetime: timestamp (nullable = true)
 |-- CellID: integer (nullable = true)
 |-- countrycode: integer (nullable = true)
 |-- smsin: double (nullable = true)
 |-- smsout: double (nullable = true)
 |-- callin: double (nullable = true)
 |-- callout: double (nullable = true)
 |-- internet: double (nullable = true)

--- Inferred Schema ---
root
 |-- datetime: timestamp (nullable = true)
 |-- CellID: integer (nullable = true)
 |-- countrycode: integer (nullable = true)
 |-- smsin: double (nullable = true)
 |-- smsout: double (nullable = true)
 |-- callin: double (nullable = true)
 |-- callout: double (nullable = true)
 |-- internet: double (nullable = true)



In [12]:
# Task 30: Import Spark functions needed to count and analyze the loaded data.

from pyspark.sql.functions import count

# Task 30: Count the total number of raw telecom records loaded from all daily files.

row_count = raw_network_df.count()

print("Total row count:", row_count)

Total row count: 15089165


In [13]:
# Task 30: Count the number of unique geographic grids using CellID.

unique_grids = raw_network_df.select("CellID").distinct().count()

print("Unique grids:", unique_grids)

Unique grids: 10000


In [14]:
# Task 30: Count the number of distinct country-code categories in the raw data.

country_code_categories = (
    raw_network_df
    .select("countrycode")
    .distinct()
    .count()
)

print("Country-code categories:", country_code_categories)

Country-code categories: 326


In [15]:
# Task 30: Count the number of distinct hourly timestamp intervals in the raw data.

distinct_hourly_intervals = (
    raw_network_df
    .select("datetime")
    .distinct()
    .count()
)

print("Distinct hourly intervals:", distinct_hourly_intervals)

Distinct hourly intervals: 168


In [16]:
# Task 31: Import input_file_name to track which source file each row came from.

from pyspark.sql.functions import input_file_name

In [17]:
# Task 31: Add the source file name to every row for data traceability.

raw_network_df = raw_network_df.withColumn(
    "input_file_name",
    input_file_name()
)

In [18]:
# Task 31: Display the source file name along with sample telecom records.

raw_network_df.select(
    "datetime",
    "CellID",
    "countrycode",
    "input_file_name"
).show(5, truncate=False)

+-------------------+------+-----------+---------------------------------------------------------+
|datetime           |CellID|countrycode|input_file_name                                          |
+-------------------+------+-----------+---------------------------------------------------------+
|2013-11-01 00:00:00|1     |0          |file:///D:/NOPIS/data/sms-call-internet-mi-2013-11-01.csv|
|2013-11-01 00:00:00|1     |33         |file:///D:/NOPIS/data/sms-call-internet-mi-2013-11-01.csv|
|2013-11-01 00:00:00|1     |39         |file:///D:/NOPIS/data/sms-call-internet-mi-2013-11-01.csv|
|2013-11-01 00:00:00|2     |0          |file:///D:/NOPIS/data/sms-call-internet-mi-2013-11-01.csv|
|2013-11-01 00:00:00|2     |33         |file:///D:/NOPIS/data/sms-call-internet-mi-2013-11-01.csv|
+-------------------+------+-----------+---------------------------------------------------------+
only showing top 5 rows


In [19]:
# Task 31: Count the number of unique source files loaded into the DataFrame.

source_file_count = (
    raw_network_df
    .select("input_file_name")
    .distinct()
    .count()
)

print("Source file count:", source_file_count)

Source file count: 7


In [20]:
# Task 32: Check how many partitions Spark uses to process the distributed DataFrame.

partition_count = raw_network_df.rdd.getNumPartitions()

print("Partition count:", partition_count)

Partition count: 15


SP2


In [21]:
# Task 33: Rename the raw telecom columns to the canonical project column names.

clean_network_df = (
    raw_network_df
    .withColumnRenamed("datetime", "timestamp")
    .withColumnRenamed("CellID", "grid_id")
    .withColumnRenamed("countrycode", "country_code")
    .withColumnRenamed("smsin", "sms_in")
    .withColumnRenamed("smsout", "sms_out")
    .withColumnRenamed("callin", "call_in")
    .withColumnRenamed("callout", "call_out")
    .withColumnRenamed("internet", "internet_activity")
)

In [22]:
# Task 33: Check that all raw columns were renamed to the canonical project names.

clean_network_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- grid_id: integer (nullable = true)
 |-- country_code: integer (nullable = true)
 |-- sms_in: double (nullable = true)
 |-- sms_out: double (nullable = true)
 |-- call_in: double (nullable = true)
 |-- call_out: double (nullable = true)
 |-- internet_activity: double (nullable = true)
 |-- input_file_name: string (nullable = false)



In [23]:
# Task 34: Cast the timestamp and activity columns to usable Spark data types.

from pyspark.sql.functions import col

clean_network_df = (
    clean_network_df
    .withColumn("timestamp", col("timestamp").cast("timestamp"))
    .withColumn("sms_in", col("sms_in").cast("double"))
    .withColumn("sms_out", col("sms_out").cast("double"))
    .withColumn("call_in", col("call_in").cast("double"))
    .withColumn("call_out", col("call_out").cast("double"))
    .withColumn("internet_activity", col("internet_activity").cast("double"))
)

In [24]:
# Task 34: Verify that the timestamp and activity columns have the correct data types.

clean_network_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- grid_id: integer (nullable = true)
 |-- country_code: integer (nullable = true)
 |-- sms_in: double (nullable = true)
 |-- sms_out: double (nullable = true)
 |-- call_in: double (nullable = true)
 |-- call_out: double (nullable = true)
 |-- internet_activity: double (nullable = true)
 |-- input_file_name: string (nullable = false)



In [25]:
# Task 34: Count distinct timestamps to verify that the expected hourly cadence is preserved.

distinct_timestamps = (
    clean_network_df
    .select("timestamp")
    .distinct()
    .count()
)

print("Distinct timestamps:", distinct_timestamps)

Distinct timestamps: 168


In [26]:
# Task 34: Verify that the number of distinct timestamps equals the expected number of daily files times 24 hours.

expected_timestamps = source_file_count * 24

print("Expected hourly intervals:", expected_timestamps)
print("Actual hourly intervals:", distinct_timestamps)

Expected hourly intervals: 168
Actual hourly intervals: 168


In [27]:
# Task 34: Verify that the number of distinct timestamps equals the expected number of daily files times 24 hours.

expected_timestamps = source_file_count * 24

print("Expected hourly intervals:", expected_timestamps)
print("Actual hourly intervals:", distinct_timestamps)

Expected hourly intervals: 168
Actual hourly intervals: 168


In [28]:
# Task 35: Create a list of the individual network activity columns to validate and profile.

activity_columns = [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]

In [29]:
# Task 35: Count all records before applying validation and cleaning rules.

rows_before_cleaning = clean_network_df.count()

print("Rows before cleaning:", rows_before_cleaning)

Rows before cleaning: 15089165


In [30]:
# Task 35: Define the condition that identifies rows with missing identifiers or negative activity values.

from pyspark.sql.functions import coalesce, col, lit

invalid_condition = (
    col("grid_id").isNull()
    | col("timestamp").isNull()
    | (coalesce(col("sms_in"), lit(0.0)) < 0)
    | (coalesce(col("sms_out"), lit(0.0)) < 0)
    | (coalesce(col("call_in"), lit(0.0)) < 0)
    | (coalesce(col("call_out"), lit(0.0)) < 0)
    | (coalesce(col("internet_activity"), lit(0.0)) < 0)
)

In [31]:
# Task 35: Quarantine invalid rows instead of mixing them with trusted network data.

rejected_network_df = clean_network_df.filter(invalid_condition)

In [32]:
# Task 35: Count how many records were rejected during validation.

rejected_row_count = rejected_network_df.count()

print("Rejected rows:", rejected_row_count)

Rejected rows: 0


In [33]:
# Task 35: Remove quarantined invalid rows and fill valid activity nulls with zero.

clean_network_df = (
    clean_network_df
    .filter(~invalid_condition)
    .na.fill(0.0, subset=activity_columns)
)

In [34]:
# Task 35: Count null values in each network activity column before applying the curated-layer null-to-zero rule.

from pyspark.sql.functions import sum, when

null_report = clean_network_df.select(
    *[
        sum(
            when(col(column).isNull(), 1).otherwise(0)
        ).alias(column)
        for column in activity_columns
    ]
)

null_report.show()

+------+-------+-------+--------+-----------------+
|sms_in|sms_out|call_in|call_out|internet_activity|
+------+-------+-------+--------+-----------------+
|     0|      0|      0|       0|                0|
+------+-------+-------+--------+-----------------+



In [35]:
# Task 35: Calculate the total number of activity null values that will be handled in the curated layer.

import builtins

null_counts = null_report.first()

nulls_handled_count = builtins.sum(
    (null_counts[column] or 0)
    for column in activity_columns
)

print("Total activity null values handled:", nulls_handled_count)

Total activity null values handled: 0


In [36]:
# Task 35: Check whether any activity columns contain NaN values.

from pyspark.sql.functions import isnan, col

clean_network_df.select(
    *[
        sum(
            when(isnan(col(column)), 1).otherwise(0)
        ).alias(column)
        for column in activity_columns
    ]
).show()

+------+-------+-------+--------+-----------------+
|sms_in|sms_out|call_in|call_out|internet_activity|
+------+-------+-------+--------+-----------------+
|     0|      0|      0|       0|                0|
+------+-------+-------+--------+-----------------+



In [37]:
# Task 36: Create total SMS, total calls, and total activity while keeping the original activity measures.

from pyspark.sql.functions import col

clean_network_df = (
    clean_network_df
    .withColumn(
        "total_sms",
        col("sms_in") + col("sms_out")
    )
    .withColumn(
        "total_calls",
        col("call_in") + col("call_out")
    )
    .withColumn(
        "total_activity",
        col("sms_in")
        + col("sms_out")
        + col("call_in")
        + col("call_out")
        + col("internet_activity")
    )
)

In [38]:
# Task 36: Display the original activity measures together with the new composite indicators.

clean_network_df.select(
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity",
    "total_sms",
    "total_calls",
    "total_activity"
).show(5, truncate=False)

+------+-------+-------+--------+-----------------+---------+-----------+-------------------+
|sms_in|sms_out|call_in|call_out|internet_activity|total_sms|total_calls|total_activity     |
+------+-------+-------+--------+-----------------+---------+-----------+-------------------+
|0.3521|0.0    |0.0    |0.0273  |0.0              |0.3521   |0.0273     |0.3794             |
|0.0   |0.0    |0.0    |0.0     |0.0261           |0.0      |0.0        |0.0261             |
|1.7322|1.1047 |0.5919 |0.402   |57.7729          |2.8369   |0.9939     |61.6037            |
|0.3581|0.0    |0.0    |0.0273  |0.0              |0.3581   |0.0273     |0.38539999999999996|
|0.0   |0.0    |0.0    |0.0     |0.0274           |0.0      |0.0        |0.0274             |
+------+-------+-------+--------+-----------------+---------+-----------+-------------------+
only showing top 5 rows


In [39]:
# Task 37: Import Spark functions used to extract date and time information from the timestamp.

from pyspark.sql.functions import to_date, hour, date_format

In [40]:
# Task 37: Derive date, hour, and day_of_week from the network activity timestamp.

clean_network_df = (
    clean_network_df
    .withColumn("date", to_date("timestamp"))
    .withColumn("hour", hour("timestamp"))
    .withColumn("day_of_week", date_format("timestamp", "EEEE"))
)

In [41]:
# Task 37: Display the timestamp together with the newly derived time columns.

clean_network_df.select(
    "timestamp",
    "date",
    "hour",
    "day_of_week"
).show(10, truncate=False)

+-------------------+----------+----+-----------+
|timestamp          |date      |hour|day_of_week|
+-------------------+----------+----+-----------+
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
|2013-11-01 00:00:00|2013-11-01|0   |Friday     |
+-------------------+----------+----+-----------+
only showing top 10 rows


In [42]:
# Task 38: Count the number of trusted records remaining after cleaning and validation.

rows_after_cleaning = clean_network_df.count()

print("Rows after cleaning:", rows_after_cleaning)

Rows after cleaning: 15089165


In [43]:
# Task 38: Compare record counts before and after cleaning and report rejected rows and handled null values.

print("----- SP2 Cleaning Summary -----")

print("Rows before cleaning:", rows_before_cleaning)
print("Rows after cleaning:", rows_after_cleaning)
print("Rejected rows:", rejected_row_count)
print("Activity null values handled:", nulls_handled_count)

----- SP2 Cleaning Summary -----
Rows before cleaning: 15089165
Rows after cleaning: 15089165
Rejected rows: 0
Activity null values handled: 0


In [44]:
# Task 39: Import Spark functions needed to group country-code records and sum activity values.

from pyspark.sql import functions as F

In [45]:
# Task 39: Collapse country-code-level rows into one row per timestamp and grid_id.

grid_hour_df = (
    clean_network_df
    .groupBy("timestamp", "grid_id")
    .agg(
        F.sum("sms_in").alias("sms_in"),
        F.sum("sms_out").alias("sms_out"),
        F.sum("call_in").alias("call_in"),
        F.sum("call_out").alias("call_out"),
        F.sum("internet_activity").alias("internet_activity")
    )
)

In [46]:
# Task 39: Display the consolidated network activity at timestamp and grid level.

grid_hour_df.show(5, truncate=False)

+-------------------+-------+------------------+-------+-------+------------------+------------------+
|timestamp          |grid_id|sms_in            |sms_out|call_in|call_out          |internet_activity |
+-------------------+-------+------------------+-------+-------+------------------+------------------+
|2013-11-01 00:00:00|58     |0.8967999999999999|0.4971 |0.5936 |1.0839999999999999|62.6787           |
|2013-11-01 00:00:00|391    |0.2297            |0.0743 |0.2076 |0.0971            |11.272599999999999|
|2013-11-01 00:00:00|420    |0.4202            |0.315  |0.0729 |0.2469            |22.3749           |
|2013-11-01 00:00:00|602    |0.7717            |0.867  |0.0939 |0.1188            |21.5292           |
|2013-11-01 00:00:00|761    |7.0189            |5.1264 |1.6685 |2.9327            |170.0255          |
+-------------------+-------+------------------+-------+-------+------------------+------------------+
only showing top 5 rows


In [47]:
# Task 39: Verify that country_code was removed after aggregation to grid and hourly grain.

print(grid_hour_df.columns)

['timestamp', 'grid_id', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet_activity']


In [48]:
# Task 40: Create total SMS, total calls, and total activity for each grid and hourly timestamp.

grid_hour_df = (
    grid_hour_df
    .withColumn(
        "total_sms",
        F.col("sms_in") + F.col("sms_out")
    )
    .withColumn(
        "total_calls",
        F.col("call_in") + F.col("call_out")
    )
    .withColumn(
        "total_activity",
        F.col("sms_in")
        + F.col("sms_out")
        + F.col("call_in")
        + F.col("call_out")
        + F.col("internet_activity")
    )
)

In [49]:
# Task 40: Display the original activity measures together with the calculated total activity measures.

grid_hour_df.select(
    "timestamp",
    "grid_id",
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity",
    "total_sms",
    "total_calls",
    "total_activity"
).show(5, truncate=False)

+-------------------+-------+------------------+-------+-------+------------------+------------------+------------------+-------------------+-----------------+
|timestamp          |grid_id|sms_in            |sms_out|call_in|call_out          |internet_activity |total_sms         |total_calls        |total_activity   |
+-------------------+-------+------------------+-------+-------+------------------+------------------+------------------+-------------------+-----------------+
|2013-11-01 00:00:00|58     |0.8967999999999999|0.4971 |0.5936 |1.0839999999999999|62.6787           |1.3939            |1.6776             |65.75019999999999|
|2013-11-01 00:00:00|391    |0.2297            |0.0743 |0.2076 |0.0971            |11.272599999999999|0.304             |0.3047             |11.8813          |
|2013-11-01 00:00:00|420    |0.4202            |0.315  |0.0729 |0.2469            |22.3749           |0.7352000000000001|0.31980000000000003|23.4299          |
|2013-11-01 00:00:00|602    |0.7717     

In [50]:
# Task 40: Create a date column from the hourly timestamp for daily grid-level aggregation.

daily_source_df = grid_hour_df.withColumn(
    "date",
    F.to_date("timestamp")
)

In [51]:
# Task 40: Calculate total daily network activity for each geographic grid.

daily_traffic_summary = (
    daily_source_df
    .groupBy("date", "grid_id")
    .agg(
        F.sum("total_activity").alias("daily_activity")
    )
)

In [52]:
# Task 40: Display the daily total activity for each geographic grid.

daily_traffic_summary.show(5, truncate=False)

+----------+-------+------------------+
|date      |grid_id|daily_activity    |
+----------+-------+------------------+
|2013-11-01|3662   |11906.305699999999|
|2013-11-01|892    |373.1352          |
|2013-11-01|2415   |1765.9360999999997|
|2013-11-01|3406   |329.47240000000005|
|2013-11-01|3924   |6829.906400000003 |
+----------+-------+------------------+
only showing top 5 rows


In [53]:
# Task 41: Calculate total activity for each grid across the selected time window.

grid_activity_ranking = (
    grid_hour_df
    .groupBy("grid_id")
    .agg(
        F.sum("total_activity").alias("window_total_activity")
    )
)

In [54]:
# Task 41: Identify the top 10 grids with the highest total activity.

hotspot_ranking = (
    grid_activity_ranking
    .orderBy(F.desc("window_total_activity"))
    .limit(10)
)

In [55]:
# Task 41: Display the top 10 high-activity grids.

hotspot_ranking.show(truncate=False)

+-------+---------------------+
|grid_id|window_total_activity|
+-------+---------------------+
|5161   |1789842.5790000004   |
|5059   |1653950.4648000002   |
|5259   |1429056.1319999998   |
|5061   |1381251.593          |
|6064   |1280323.972          |
|5258   |1252701.0197         |
|5159   |1246998.1886000005   |
|4459   |1154504.9723         |
|5262   |1153148.4885         |
|5758   |1150307.1738999998   |
+-------+---------------------+



In [57]:
# Task 42: Extract the hour from each timestamp to find the busiest hour of the day.

peak_hour_df = grid_hour_df.withColumn(
    "hour",
    F.hour("timestamp")
)

In [61]:
peak_hour_df.show()

+-------------------+-------+------------------+------------------+-------+-------------------+------------------+------------------+-------------------+------------------+----+
|          timestamp|grid_id|            sms_in|           sms_out|call_in|           call_out| internet_activity|         total_sms|        total_calls|    total_activity|hour|
+-------------------+-------+------------------+------------------+-------+-------------------+------------------+------------------+-------------------+------------------+----+
|2013-11-01 00:00:00|     58|0.8967999999999999|            0.4971| 0.5936| 1.0839999999999999|           62.6787|            1.3939|             1.6776| 65.75019999999999|   0|
|2013-11-01 00:00:00|    391|            0.2297|            0.0743| 0.2076|             0.0971|11.272599999999999|             0.304|             0.3047|           11.8813|   0|
|2013-11-01 00:00:00|    420|            0.4202|             0.315| 0.0729|             0.2469|           22.3

In [58]:
# Task 42: Sum total network activity across all grids for each hour of the day.

hourly_activity = (
    peak_hour_df
    .groupBy("hour")
    .agg(
        F.sum("total_activity").alias("hourly_total_activity")
    )
)

In [59]:
# Task 42: Identify the hour with the highest total network activity.

peak_activity_hour = (
    hourly_activity
    .orderBy(F.desc("hourly_total_activity"))
    .limit(1)
)

In [60]:
# Task 42: Display the peak network activity hour.

peak_activity_hour.show()

+----+---------------------+
|hour|hourly_total_activity|
+----+---------------------+
|  17| 4.7540309663200006E7|
+----+---------------------+



In [62]:
# Task 43: Calculate the percentage of total network activity contributed by internet activity.

grid_hour_df = grid_hour_df.withColumn(
    "internet_share",
    F.when(
        F.col("total_activity") > 0,
        (F.col("internet_activity") / F.col("total_activity")) * 100
    ).otherwise(0)
)

In [63]:
# Task 43: Display internet activity, total activity, and internet share for each grid and hour.

grid_hour_df.select(
    "timestamp",
    "grid_id",
    "internet_activity",
    "total_activity",
    "internet_share"
).show(5, truncate=False)

+-------------------+-------+------------------+-----------------+-----------------+
|timestamp          |grid_id|internet_activity |total_activity   |internet_share   |
+-------------------+-------+------------------+-----------------+-----------------+
|2013-11-01 00:00:00|58     |62.6787           |65.75019999999999|95.32853132005683|
|2013-11-01 00:00:00|391    |11.272599999999999|11.8813          |94.87682324324778|
|2013-11-01 00:00:00|420    |22.3749           |23.4299          |95.49720656084747|
|2013-11-01 00:00:00|602    |21.5292           |23.3806          |92.08146925228607|
|2013-11-01 00:00:00|761    |170.0255          |186.772          |91.0337202578545 |
+-------------------+-------+------------------+-----------------+-----------------+
only showing top 5 rows


In [64]:
# Task 44: Create hourly_grid_summary with exactly one record per grid_id and timestamp.

hourly_grid_summary = grid_hour_df

In [65]:
# Task 44: Verify that country_code does not exist in hourly_grid_summary.

print(hourly_grid_summary.columns)

['timestamp', 'grid_id', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet_activity', 'total_sms', 'total_calls', 'total_activity', 'internet_share']


In [66]:
# Task 44: Count duplicate records based on the canonical grid_id and timestamp grain.

duplicate_count = (
    hourly_grid_summary
    .groupBy("grid_id", "timestamp")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate grid-hour combinations:", duplicate_count)

Duplicate grid-hour combinations: 0


In [67]:
# Task 44: Fail the Spark job if duplicate grid_id and timestamp combinations exist.

assert duplicate_count == 0, (
    f"SP3 validation failed: found {duplicate_count} duplicate "
    "grid_id and timestamp combinations."
)

print("SP3 duplicate validation: PASSED")

SP3 duplicate validation: PASSED


In [68]:
# Task 44: Verify that hourly_grid_summary has fewer rows than the country-code-level clean data.

clean_row_count = clean_network_df.count()

hourly_summary_row_count = hourly_grid_summary.count()

print("clean_network_df rows:", clean_row_count)
print("hourly_grid_summary rows:", hourly_summary_row_count)

clean_network_df rows: 15089165
hourly_grid_summary rows: 1679994


In [69]:
# Task 44: Verify that the hourly summary does not exceed the maximum possible grid-hour combinations.

maximum_possible_rows = source_file_count * 24 * 10000

print("Maximum possible rows:", maximum_possible_rows)
print("Actual hourly summary rows:", hourly_summary_row_count)

assert hourly_summary_row_count <= maximum_possible_rows, (
    "SP3 validation failed: hourly_grid_summary exceeds the maximum possible "
    "number of grid and hourly timestamp combinations."
)

print("Maximum row count validation: PASSED")

Maximum possible rows: 1680000
Actual hourly summary rows: 1679994
Maximum row count validation: PASSED
